In [1]:
import torch
import math
import os
torch.manual_seed(35)

In [2]:
corpus = '''
the cat sat on the mat
the dog sat on the log
the cat ate the rat
the rat ran from the cat
the dog ran after the rat
the bird flew over the tree
the squirrel climbed the tree
the rabbit hopped in the field
the fox chased the rabbit
the owl perched on the branch
the mouse scurried under the bush
the frog leaped into the pond
the duck swam in the lake
the goose honked loudly
the hawk soared in the sky
the bear lumbered through the forest
the deer grazed in the meadow
the wolf howled at the moon
the snake slithered in the grass
the lizard basked in the sun
the turtle rested on the rock
the butterfly fluttered from flower to flower
the bee buzzed around the hive
the ant marched in a line
the grasshopper chirped in the tall grass
the dragonfly darted over the water
the moth fluttered near the light
the spider spun its web
the worm crawled in the soil
the snail left a slimy trail
'''

words = corpus.split()
vocab = sorted(set(words))
word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {val: key for key, val in word_to_index.items()}
vocab_size = len(vocab)
vocab_size

96

In [3]:
embed_dim = 128
number_of_heads = 8
head_size = embed_dim//number_of_heads  
ffn_hidden = 512

In [4]:
# all embeddings
embedding_table = torch.nn.Parameter(torch.randn(vocab_size, embed_dim))

# decoder self attention weights
dec_wq = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))
dec_wk = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))
dec_wv = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))
dec_wo = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))  

# decoder FFN
dec_ff1_w = torch.nn.Parameter(torch.randn(embed_dim, ffn_hidden))
dec_ff1_b = torch.nn.Parameter(torch.zeros(ffn_hidden))
dec_ff2_w = torch.nn.Parameter(torch.randn(ffn_hidden, embed_dim))
dec_ff2_b = torch.nn.Parameter(torch.zeros(embed_dim))

# Logits function weights
out_w = torch.nn.Parameter(torch.randn(embed_dim, vocab_size))
out_b = torch.nn.Parameter(torch.zeros(vocab_size))

# normalization layers
dec_norm1 = torch.nn.LayerNorm(embed_dim)
dec_norm2 = torch.nn.LayerNorm(embed_dim)

all_params = [
    embedding_table,
    dec_wq, dec_wk, dec_wv, dec_wo,
    dec_ff1_w, dec_ff1_b, dec_ff2_w, dec_ff2_b,
    out_w, out_b,
    *dec_norm1.parameters(),
    *dec_norm2.parameters(),
]

optimizer = torch.optim.Adam(all_params, lr=0.001)
loss_fn   = torch.nn.CrossEntropyLoss()

In [5]:
def pos_enc(seq_len):
    pe = torch.zeros(seq_len, embed_dim)
    for pos in range(seq_len):
        for i in range(0, embed_dim, 2):
            pe[pos, i]   = math.sin(pos / 10000 ** (i / embed_dim))
            pe[pos, i+1] = math.cos(pos / 10000 ** (i / embed_dim))
    return pe

In [6]:
training_sequences = []

for line in corpus.strip().split('\n'):
    print(line)
    w = line.split()
    source_token = torch.tensor([word_to_index[x] for x in w[:-1]])
    target_token = torch.tensor([word_to_index[x] for x in w[1:]])
    training_sequences.append((source_token, target_token))
    print(source_token, target_token)
    
print(f'Training sequences: {len(training_sequences)}')

the cat sat on the mat
tensor([85, 14, 70, 60, 85]) tensor([14, 70, 60, 85, 54])
the dog sat on the log
tensor([85, 21, 70, 60, 85]) tensor([21, 70, 60, 85, 50])
the cat ate the rat
tensor([85, 14,  5, 85]) tensor([14,  5, 85, 67])
the rat ran from the cat
tensor([85, 67, 66, 31, 85]) tensor([67, 66, 31, 85, 14])
the dog ran after the rat
tensor([85, 21, 66,  1, 85]) tensor([21, 66,  1, 85, 67])
the bird flew over the tree
tensor([85,  9, 25, 61, 85]) tensor([ 9, 25, 61, 85, 89])
the squirrel climbed the tree
tensor([85, 81, 17, 85]) tensor([81, 17, 85, 89])
the rabbit hopped in the field
tensor([85, 65, 39, 41, 85]) tensor([65, 39, 41, 85, 24])
the fox chased the rabbit
tensor([85, 29, 15, 85]) tensor([29, 15, 85, 65])
the owl perched on the branch
tensor([85, 62, 63, 60, 85]) tensor([62, 63, 60, 85, 10])
the mouse scurried under the bush
tensor([85, 58, 71, 91, 85]) tensor([58, 71, 91, 85, 11])
the frog leaped into the pond
tensor([85, 30, 45, 42, 85]) tensor([30, 45, 42, 85, 64])
th

In [7]:
# load existing weights if available
if os.path.exists('decoder_only_weights.pth'):
    checkpoint = torch.load('gpt_weights.pth')
    embedding_table.data = checkpoint['embedding_table'].data
    dec_wq.data = checkpoint['dec_wq'].data
    dec_wk.data = checkpoint['dec_wk'].data
    dec_wv.data = checkpoint['dec_wv'].data
    dec_wo.data = checkpoint['dec_wo'].data
    dec_ff1_w.data = checkpoint['dec_ff1_w'].data
    dec_ff1_b.data = checkpoint['dec_ff1_b'].data
    dec_ff2_w.data = checkpoint['dec_ff2_w'].data
    dec_ff2_b.data = checkpoint['dec_ff2_b'].data
    out_w.data = checkpoint['out_w'].data
    out_b.data = checkpoint['out_b'].data
    dec_norm1.load_state_dict(checkpoint['dec_norm1'])
    dec_norm2.load_state_dict(checkpoint['dec_norm2'])
    print('Loaded existing weights')
else:
    print('No saved weights, training model from scratch.')

Loaded existing weights


In [8]:
def neural_network_decoder(source_tokens):
    sequence_length = len(source_tokens)
    source_embedding = embedding_table[source_tokens] + pos_enc(sequence_length)

    #masked self attention, hides next element in series using '-inf' for prediction
    mask_self_attention = torch.triu(torch.ones(sequence_length, sequence_length), diagonal=1)
    mask_self_attention = mask_self_attention.masked_fill(mask_self_attention ==1, float('-inf'))
    
    Qd = source_embedding @ dec_wq
    Kd = source_embedding @ dec_wk
    Vd = source_embedding @ dec_wv

    # Splitting Qd,Kd, Vd to make it multi-headed 
    Qd = Qd.view(sequence_length, number_of_heads, head_size).transpose(0, 1)
    Kd = Kd.view(sequence_length, number_of_heads, head_size).transpose(0, 1)
    Vd = Vd.view(sequence_length, number_of_heads, head_size).transpose(0, 1)
    
    scores_dec = Qd @ Kd.transpose(-2, -1) / math.sqrt(head_size) + mask_self_attention
    attention_dec = torch.softmax(scores_dec, dim=-1)
    out_dec = attention_dec @ Vd 

    # Concatenating multi-head or joining them back
    out_dec = out_dec.transpose(0, 1).contiguous().view(sequence_length, embed_dim)

    #Output Projection Layer
    out_dec = out_dec @ dec_wo

    # Residual Layer1
    residual_layer1 = source_embedding + out_dec

    # Normalization Layer
    normalization_layer1 = dec_norm1(residual_layer1)

    
    #FFN
    # Linear Layer
    linear_layer1 = normalization_layer1 @ dec_ff1_w + dec_ff1_b
    relu_layer1 = torch.relu(linear_layer1)
    linear_layer2 = relu_layer1 @ dec_ff2_w + dec_ff2_b

    #Residual Layer2
    residual_layer2 = normalization_layer1 + linear_layer2

    # Normalization Layer2
    normalization_layer2 = dec_norm2(residual_layer2)

    #Logits Layer
    logits = normalization_layer2 @ out_w + out_b
    return logits


In [9]:
epochs = 200
for epoch in range(1, epochs + 1):
    total_loss = 0.0

    for source_tokens, target_tokens in training_sequences:

        logits = neural_network_decoder(source_tokens)
        # Loss Calculation Layer - Cross Entropy, This automatically converts tokens to embeddding
            # and calculates the loss
        loss = loss_fn(logits, target_tokens)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)  # prevent exploding gradients
        optimizer.step()

        total_loss += loss.item()
    avg_loss = total_loss / len(training_sequences)
    print('epoch', epoch, 'avg_loss', avg_loss)

epoch 1 avg_loss 1.0034461160500845
epoch 2 avg_loss 0.8379776219526927
epoch 3 avg_loss 0.8405600190162659
epoch 4 avg_loss 0.8301819562911987
epoch 5 avg_loss 0.8299235065778097
epoch 6 avg_loss 0.8285010655721029
epoch 7 avg_loss 0.8285189767678579
epoch 8 avg_loss 0.8277707934379578
epoch 9 avg_loss 0.8276163180669148
epoch 10 avg_loss 0.827049030860265
epoch 11 avg_loss 0.8270627001921336
epoch 12 avg_loss 0.8264173785845439
epoch 13 avg_loss 0.8259926021099091
epoch 14 avg_loss 0.8255311965942382
epoch 15 avg_loss 0.8252128839492798
epoch 16 avg_loss 0.8250483274459839
epoch 17 avg_loss 0.82526416182518
epoch 18 avg_loss 0.8246900737285614
epoch 19 avg_loss 0.851997172832489
epoch 20 avg_loss 0.8200071036815644
epoch 21 avg_loss 0.8323424200216929
epoch 22 avg_loss 0.8367045680681865
epoch 23 avg_loss 0.8269859155019125
epoch 24 avg_loss 0.8271800875663757
epoch 25 avg_loss 0.8246214071909587
epoch 26 avg_loss 0.8247780720392863
epoch 27 avg_loss 0.8230291545391083
epoch 28 avg_l

In [10]:
torch.save({
    'embedding_table': embedding_table,
    'dec_wq': dec_wq, 'dec_wk': dec_wk, 'dec_wv': dec_wv, 'dec_wo': dec_wo,
    'dec_ff1_w': dec_ff1_w, 'dec_ff1_b': dec_ff1_b,
    'dec_ff2_w': dec_ff2_w, 'dec_ff2_b': dec_ff2_b,
    'out_w': out_w, 'out_b': out_b,
    'dec_norm1': dec_norm1.state_dict(),
    'dec_norm2': dec_norm2.state_dict(),
    'word_to_index': word_to_index,
    'index_to_word': index_to_word,
    'vocab_size': vocab_size,
}, 'decoder_only_weights.pth')
print('Saved.')

Saved.


In [18]:
def generate(prompt, max_words=15, temperature = 1):
    words_in = prompt.lower().split()
    
    unknown = [w for w in words_in if w not in word_to_index]
    
    if unknown:
        print(f'Unknown words: {unknown}')
        print(f'Use words from vocab: {sorted(word_to_index.keys())}')
        return

    entered_by_user = list(words_in)
    print(entered_by_user)

    with torch.no_grad():
        for _ in range(max_words):
            tokens = torch.tensor([word_to_index[w] for w in entered_by_user])
            logits = neural_network_decoder(tokens)          
            #next_token = logits[-1].argmax().item() #picks out with highest probability
            probs  = torch.softmax(logits[-1] / temperature, dim=-1) # Adding temperature adds randomness
            next_token = torch.multinomial(probs, 1).item() # This pick a random number between 0 and 1 and see where it aligns with above estimated probability
            entered_by_user.append(index_to_word[next_token])
            #next_word  = index_to_word[next_token]
            #generated.append(next_word)

    print(f'Prompt   : {prompt}')
    print(f'Generated: {" ".join(entered_by_user)}')

# test
generate('the cat', temperature = 0.3)
print()
generate('the dog')
print()
generate('the cat', temperature = 3)

['the', 'cat']
Prompt   : the cat
Generated: the cat sat on the mat marched line ate rat grazed the tree over loudly the the

['the', 'dog']
Prompt   : the dog
Generated: the dog sat on the log left the on grass hopped the mat sat on the ran

['the', 'cat']
Prompt   : the cat
Generated: the cat sat on the darted loudly forest the grass fluttered leaped grazed soared the tree the
